In [1]:
pip install selenium

  Using cached selenium-4.1.0-py3-none-any.whl (958 kB)
  Using cached trio-0.19.0-py3-none-any.whl (356 kB)
  Using cached urllib3-1.26.8-py2.py3-none-any.whl (138 kB)
  Using cached trio_websocket-0.9.2-py3-none-any.whl (16 kB)
  Using cached async_generator-1.10-py3-none-any.whl (18 kB)
  Using cached sniffio-1.2.0-py3-none-any.whl (10 kB)
  Using cached outcome-1.1.0-py2.py3-none-any.whl (9.7 kB)
  Using cached wsproto-1.0.0-py3-none-any.whl (24 kB)
     |████████████████████████████████| 55 kB 2.0 MB/s 
     |████████████████████████████████| 3.6 MB 10.2 MB/s 
     |████████████████████████████████| 54 kB 3.0 MB/s 
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
requests 2.23.0 requires urllib3!=1.25

In [2]:
pip install chromedriver_binary

In [3]:
pip install mecab-python3

In [4]:
import urllib.request, urllib.error
from bs4 import BeautifulSoup
import requests
import os
import re
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import chromedriver_binary
import numpy as np
import pickle
from gensim.models import word2vec
import MeCab
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

/usr/local/lib/python3.7/dist-packages/requests/__init__.py:91: RequestsDependencyWarning: urllib3 (1.26.8) or chardet (3.0.4) doesn't match a supported version!
  RequestsDependencyWarning)


In [ ]:
class SaspenceScraiping:
    """
    wikiから事件名、放映日時、舞台、出演者、あらすじ，犯人の名前を取得する
    """
    def __init__(self, max_num):
        """
        インスタンス化した時点でスクレイピングを開始する
        サスペンスドラマデータベースにアクセスする
        取得するページ件数は1ページ目からmax_numページ目までとする。
        """
        # 取得したデータ一覧の保存場所
        self.RESULT_PATH = '/content/drive/MyDrive/PAI_PROJECT/Suspection/dataset'
        # 取得したデータ一覧名
        self.RESULT_NAME = 'data_summary.csv'
        # 取得したデータフレームの列名
        self.COLUMNS = ['id', 'drama_title', 'cast_list','synopsis']
        # 検索するページ上限数
        self.max_num = max_num + 1
        self.drama_id = ''
        self.id_num = 0
        self.df = pd.DataFrame(columns = self.COLUMNS)

        try:
            first_time = time.time()
            # サイトの形式が何パターンかあるので，パターンごとに分けて抽出する．
            # wikiの西村京太郎トラベルシリーズ34~38の3件を取得(36,37はあらすじデータが無かったため飛ばす)
            for page_num in range(26, 31):
                if page_num == 28 or page_num == 29:
                    continue
                start_time = time.time()
                self.id_num += 1
                self.drama_id = page_num + 8
                url = "https://ja.wikipedia.org/wiki/%E8%A5%BF%E6%9D%91%E4%BA%AC%E5%A4%AA%E9%83%8E%E3%83%88%E3%83%A9%E3%83%99%E3%83%AB%E3%83%9F%E3%82%B9%E3%83%86%E3%83%AA%E3%83%BC"
                html = urllib.request.urlopen(url)
                soup = BeautifulSoup(html, "lxml")
                
                links_1 = [url.get('href') for url in soup.find_all('a', class_='external text')][page_num]
                res = self.connect_url(links_1)
                soup = BeautifulSoup(res.text, 'lxml')
                
                # 一覧ページから以下の情報を取得する
                # ドラマ名の取得
                drama_title = soup.find_all('font', color="#414141")[0].text.strip()
                
                links_2 = links_1.replace("index.html","story.html")
                res = urllib.request.urlopen(links_2)
                soup = BeautifulSoup(res, 'lxml')
                
                # あらすじの取得
                synopsis = soup.find_all('font', size="2")[3].text.strip()
                
                # 登場人物の取得
                cast_list = ""

                
                self.add_df(drama_title, cast_list, synopsis)
                self.write_result(drama_title, cast_list, synopsis)

                process_time = time.time() - start_time
                print('{}件目完了：　処理時間：{:.3f}秒'.format(self.id_num, process_time))
            
            # wikiの西村京太郎トラベルシリーズ39~49の6件を取得(39,40,41,49はURLが確認できなかった．47は取得できなかった．)
            for page_num in range(31, 38):
                if page_num == 36:
                    continue
                start_time = time.time()
                self.id_num += 1
                self.drama_id = page_num + 11
                url = "https://ja.wikipedia.org/wiki/%E8%A5%BF%E6%9D%91%E4%BA%AC%E5%A4%AA%E9%83%8E%E3%83%88%E3%83%A9%E3%83%99%E3%83%AB%E3%83%9F%E3%82%B9%E3%83%86%E3%83%AA%E3%83%BC"
                html = urllib.request.urlopen(url)
                soup = BeautifulSoup(html, "lxml")
                
                links_1 = [url.get('href') for url in soup.find_all('a', class_='external text')][page_num]
                res = self.connect_url(links_1)
                soup = BeautifulSoup(res.text, 'lxml')
                
                # 一覧ページから以下の情報を取得する
                # ドラマ名の取得
                drama_title = soup.find_all('td', class_="font14")[0].text.strip()
                
                # あらすじの取得
                if page_num == 33 or page_num == 34 or page_num == 35 or page_num == 37:
                    synopsis = soup.find_all('font', color="#FFFFFF")[0].text.strip()
                else:
                    synopsis = soup.find_all('font', color="#FFFFFF")[1].text.strip()
                
                # 登場人物の取得
                cast_list = ""
                
                self.add_df(drama_title, cast_list, synopsis)
                self.write_result(drama_title, cast_list, synopsis)

                process_time = time.time() - start_time
                print('{}件目完了：　処理時間：{:.3f}秒'.format(self.id_num, process_time))
                    
            # wikiの西村京太郎トラベルシリーズ50~65の16件を取得(重複データは飛ばす)
            for page_num in range(38, self.max_num):
                if page_num == 49:
                    continue
                self.id_num += 1
                if page_num >= 50:
                    self.drama_id = page_num + 11
                else:
                    self.drama_id = page_num + 12
                start_time = time.time()
                url = "https://ja.wikipedia.org/wiki/%E8%A5%BF%E6%9D%91%E4%BA%AC%E5%A4%AA%E9%83%8E%E3%83%88%E3%83%A9%E3%83%99%E3%83%AB%E3%83%9F%E3%82%B9%E3%83%86%E3%83%AA%E3%83%BC"
                html = urllib.request.urlopen(url)
                soup = BeautifulSoup(html, "lxml")
                
                links = [url.get('href') for url in soup.find_all('a', class_='external text')][page_num]
                res = self.connect_url(links)
                soup = BeautifulSoup(res.text, 'lxml')
                
                # 一覧ページから以下の情報を取得する
                # あらすじの取得
                synopsis = soup.find_all('span', class_='font14')[1].text.rstrip()

                # 登場人物の取得
                cast = soup.find('span', class_='font14').stripped_strings
                cast_list = ','.join(list(cast))

                # ドラマ名の取得
                raw_data = soup.find_all('td', class_='font18bold')
                drama_title = [name.text for name in raw_data][0]
                
                self.add_df(drama_title, cast_list, synopsis)
                self.write_result(drama_title, cast_list, synopsis)
                

                process_time = time.time() - start_time
                print('{}件目完了：　処理時間：{:.3f}秒'.format(self.id_num, process_time))

        # wikiの西村京太郎トラベルシリーズ69~72の4件を取得
            for page_num in range(58, 62):
                self.id_num += 1
                self.drama_id = page_num + 11
                start_time = time.time()
                url = "https://ja.wikipedia.org/wiki/%E8%A5%BF%E6%9D%91%E4%BA%AC%E5%A4%AA%E9%83%8E%E3%83%88%E3%83%A9%E3%83%99%E3%83%AB%E3%83%9F%E3%82%B9%E3%83%86%E3%83%AA%E3%83%BC"
                html = urllib.request.urlopen(url)
                soup = BeautifulSoup(html, "lxml")
                
                links = [url.get('href') for url in soup.find_all('a', class_='external text')][58]
                # res = connect_url(links)
                html = urllib.request.urlopen(links)
                soup = BeautifulSoup(html, 'lxml')
                
                synopsis = soup.find_all('div', class_="txt")[0].text.rstrip()

                cast = soup.find_all('div', class_="txt")[1].text
                cast_list = ','.join(list(cast))

                raw_data = soup.find_all('p', class_='ttl')
                drama_title = [name.text for name in raw_data][0]
                self.add_df(drama_title, cast_list, synopsis)
                self.write_result(drama_title, cast_list, synopsis)
                
                process_time = time.time() - start_time
                print('{}件目完了：　処理時間：{:.3f}秒'.format(self.id_num, process_time))

        except requests.exceptions.HTTPError as e:
            print(e)
        except requests.exceptions.ConnectTimeout as e:
            print(e)
        finally:
            end_time = time.time() - first_time
            print('終了、処理時間：{:.3f}秒'.format(end_time))


    def connect_url(self, target_url):
        """
        対象のURLにアクセスする関数
        アクセスできない等のエラーが発生したら例外を投げる
        """
        # 接続確立の待機時間、応答待機時間を10秒とし、それぞれの値を超えた場合は例外が発生（ConnectTimeout）
        data = requests.get(target_url, timeout=30)
        data.encoding = data.apparent_encoding
        # アクセス過多を避けるため、2秒スリープ
        time.sleep(2)

        # レスポンスのステータスコードが正常(200番台)以外の場合は、例外を発生させる(HTTPError)
        if data.status_code == requests.codes.ok:
            return data
        else:
            data.raise_for_status()


    def write_result(self, title, cast, synopsis):
        """
        スクレイピング対象となるドラマ一覧をログとして出力する関数
        """
        file_path = os.path.join(self.RESULT_PATH, self.RESULT_NAME)
        with open(file_path, mode='a', encoding='utf-8') as f:
            f.write('{}, 情報：{}, あらすじ：{}\n'.format(self.drama_id, title, cast, synopsis))


    def add_df(self, title, cast, synopsis):
        """
        取得したあらすじデータをデータフレームに格納する関数
        """
        se = pd.Series([self.drama_id, title, cast, synopsis], self.COLUMNS)
        self.df = self.df.append(se, self.COLUMNS)


if __name__ == '__main__':
    result = SaspenceScraiping(54) # 西村京太郎トラベルシリーズ34~72(29作品)の抽出
    result.df.to_csv('/content/drive/MyDrive/PAI_PROJECT/Suspection/dataset/synopsis_34-72.csv');
    print(result.df)

In [5]:
# df_synopsis_1, df_series_1は小原さんに作ってもらったデータフレーム

df_series = pd.read_csv("/content/drive/MyDrive/PAI_PROJECT/Suspection/dataset/nishimura_series_data_new.csv")
df_series = df_series.rename(columns={'episode': 'id'})
df_synopsis_1 = pd.read_excel("/content/drive/MyDrive/PAI_PROJECT/Suspection/dataset/nishimura_story_data.xlsx")
df_synopsis_1 = df_synopsis_1.rename(columns={'title': 'drama_title',
                                             'story': 'synopsis',
                                              'episode': 'id',
                                              'Unnamed: 3': 'informal'
                                             })
    
df_synopsis_2 = pd.read_csv("/content/drive/MyDrive/PAI_PROJECT/Suspection/dataset/synopsis_34-72.csv").drop('Unnamed: 0', axis=1)


display(df_series.head())
display(df_synopsis_1.head())
display(df_synopsis_2.head())

,id,character,actor,criminal
0,1,亀井刑事,愛川欽也,0
1,1,十津川警部,三橋達也,0
2,1,宮本孝,中島久之,0
3,1,安田章,村上正次,0
4,1,川島史郎,保積ぺぺ,0


,id,drama_title,synopsis,informal
0,1,終着駅〈ターミナル〉殺人事件\n上野-青森 ミステリー特急ゆうづる,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...,NaN
1,2,再婚旅行殺人事件\n出雲で死んだ女,京都で”ひさこ”という小さなバーを経営している深尾尚子（大谷直子）は\n3年前に別れた井村正...,非公式
2,3,あずさ3号殺人事件\n京都-信州 婚前クイズ旅行のワナ,午前８時新宿発の”あずさ3号”が１１時４７分に松本駅に着いた時、\n列車のトイレから男の刺殺...,非公式
3,4,寝台特急あかつき殺人事件\n東京-九州 日本縦断!ブルートレインの謎と白鷺の女,元警視庁刑事の田辺淳探偵(高岡健二)が、坂口由美子(浅野ゆう子)と寝台特急あかつき３号に乗っ...,非公式
4,5,東北新幹線殺人事件\n東京-仙台-盛岡 “やまびこ27号”死体を乗せて東日本縦断!!七夕まつりの女,東京第一生命保険に勤めるＯＬの大矢けい子が殺害された。\n亀井（愛川欽也）と西本（森本レオ）...,NaN


,id,drama_title,cast_list,synopsis
0,34,西村京太郎の\n トラベルミステリースペシャル\n 「妻を誘拐さ...,NaN,その日、警視庁捜査一課の亀井刑事（愛川欽也）は姪の結婚式に出席するため、早朝の東北新幹線に乗...
1,35,「西村京太郎トラベルミステリー 宗谷本線殺人事件」,NaN,警視庁の亀井刑事（愛川欽也）は、犯人護送のため極寒の地、北海道の稚内へとやって来た。そして無...
2,38,土曜ワイド劇場\n ...,NaN,金融会社を経営する資産家の五十嵐恭（伊武雅刀）が、自宅マンションで死体で発見された。警視庁捜...
3,42,西村京太郎トラベルミステリー 「北帰行殺人事件」,NaN,警視庁捜査一課に配属になったばかりの橋本刑事（賀集利樹）が、家庭の事情で突然辞表を提出し、郷...
4,43,「愛と殺意の津軽三味線」,NaN,京王線つつじヶ丘駅近くの小公園で、頭を鈍器で殴られた男の死体が発見され、警視庁捜査一課の十津...


In [6]:
df = pd.concat([df_synopsis_1, df_synopsis_2]).reset_index(drop=True)
df["suspect"] = "" # 犯罪者(suspect)情報を付け加える
for i in range(1,73):
    x = df_series[(df_series.id == i) & (df_series.criminal == 1)]["character"]
    y = ""
    for j in range(len(x)):
        y += x.iloc[j] + ","
    df.loc[df.id==i, "suspect"] = y
    
df["synopsis_real_criminal"] = df["synopsis"]+ "犯人は、" + df["suspect"] + "!!!"
df.to_csv('/content/drive/MyDrive/PAI_PROJECT/Suspection/dataset/nishimura_episode_data.csv')
display(df.head())

,id,drama_title,synopsis,informal,cast_list,suspect,synopsis_real_criminal
0,1,終着駅〈ターミナル〉殺人事件\n上野-青森 ミステリー特急ゆうづる,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...,NaN,NaN,"町田隆夫,",青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...
1,2,再婚旅行殺人事件\n出雲で死んだ女,京都で”ひさこ”という小さなバーを経営している深尾尚子（大谷直子）は\n3年前に別れた井村正...,非公式,NaN,"井村正義,",京都で”ひさこ”という小さなバーを経営している深尾尚子（大谷直子）は\n3年前に別れた井村正...
2,3,あずさ3号殺人事件\n京都-信州 婚前クイズ旅行のワナ,午前８時新宿発の”あずさ3号”が１１時４７分に松本駅に着いた時、\n列車のトイレから男の刺殺...,非公式,NaN,"加藤昌彦,",午前８時新宿発の”あずさ3号”が１１時４７分に松本駅に着いた時、\n列車のトイレから男の刺殺...
3,4,寝台特急あかつき殺人事件\n東京-九州 日本縦断!ブルートレインの謎と白鷺の女,元警視庁刑事の田辺淳探偵(高岡健二)が、坂口由美子(浅野ゆう子)と寝台特急あかつき３号に乗っ...,非公式,NaN,"坂口由美子,坂口文子,",元警視庁刑事の田辺淳探偵(高岡健二)が、坂口由美子(浅野ゆう子)と寝台特急あかつき３号に乗っ...
4,5,東北新幹線殺人事件\n東京-仙台-盛岡 “やまびこ27号”死体を乗せて東日本縦断!!七夕まつりの女,東京第一生命保険に勤めるＯＬの大矢けい子が殺害された。\n亀井（愛川欽也）と西本（森本レオ）...,NaN,NaN,"二宮由美,青木琢二,",東京第一生命保険に勤めるＯＬの大矢けい子が殺害された。\n亀井（愛川欽也）と西本（森本レオ）...


In [9]:
df_series.id == 72

0      False
1      False
2      False
3      False
4      False
       ...  
741     True
742     True
743     True
744     True
745     True
Name: id, Length: 746, dtype: bool

In [10]:
# df_seriesにsynopsis_plus_cast_infoを追加

df_series["synopsis"] = ""

for i in range(1,73):
    if i in df["id"].unique().tolist():
        df_series.loc[df_series["id"]==i, "synopsis"] = df.loc[df.id==i, "synopsis"].iloc[-1]

        
df_series["synopsis_plus_cast_info"] = "" # 犯罪者(suspect)情報(cast)を付け加える
df_series["synopsis_plus_cast_info"] = df_series["synopsis"]+ "犯人は、" + df_series["character"] + "!!!"
df_series.to_csv('/content/drive/MyDrive/PAI_PROJECT/Suspection/dataset/nishimura_data.csv')
display(df_series.head())

,id,character,actor,criminal,synopsis,synopsis_plus_cast_info
0,1,亀井刑事,愛川欽也,0,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...
1,1,十津川警部,三橋達也,0,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...
2,1,宮本孝,中島久之,0,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...
3,1,安田章,村上正次,0,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...
4,1,川島史郎,保積ぺぺ,0,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...,青森県立F高校の卒業生で校内新聞の編集長だった宮本孝は、卒業する時に交わした「7年経ったら皆...


In [12]:
df_series.id

0       1
1       1
2       1
3       1
4       1
       ..
741    72
742    72
743    72
744    72
745    72
Name: id, Length: 746, dtype: int64